In [18]:
# Kurulum ve ortam değişkenleri
import os
import json
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter, FieldCondition, MatchValue, Range,
    Prefetch, FusionQuery, Fusion, Document,
)
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import torch

load_dotenv()

KOLEKSIYON = "yks_named"
GPT_MODEL = "gpt-4o-mini"
device = "cuda" if torch.cuda.is_available() else "cpu"

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print(f"Cihaz: {device}, Koleksiyon: {KOLEKSIYON}, GPT: {GPT_MODEL}")

Cihaz: cpu, Koleksiyon: yks_named, GPT: gpt-4o-mini


In [19]:
# Qdrant ve embedding modeli bağlantısı
qdrant = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    port=443, https=True, timeout=60, prefer_grpc=False,
)
model = SentenceTransformer("BAAI/bge-m3", device=device)

sayi = qdrant.count(collection_name=KOLEKSIYON).count
print(f"Qdrant OK — '{KOLEKSIYON}': {sayi} point")
print(f"Model: BAAI/bge-m3, boyut: {model.get_embedding_dimension()}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6307.20it/s]


Qdrant OK — 'yks_named': 21493 point
Model: BAAI/bge-m3, boyut: 1024


In [20]:
# Jina v3.5 listwise reranker — fusion sonrası içerik bazlı yeniden sıralama
from transformers import AutoModel

reranker = AutoModel.from_pretrained(
    "jinaai/jina-reranker-v3.5",
    dtype="auto",
    trust_remote_code=True,
)
reranker.eval()
reranker.to(device)
print("Jina reranker v3.5 yüklendi")

Loading weights: 100%|██████████| 312/312 [00:00<00:00, 1019.62it/s]


Jina reranker v3.5 yüklendi


In [21]:
# Reranker API çıktı yapısını doğrula (rerank_et'i buna göre yazacağız)
test = reranker.rerank(
    "tıp",
    ["Tıp - İstanbul Üniversitesi", "Hukuk - Ankara Üniversitesi", "Tıp - Hacettepe"],
    top_n=3,
)
print("Dönen yapı:")
for r in test:
    print(r)

Dönen yapı:
{'document': 'Tıp - Hacettepe', 'relevance_score': np.float32(0.21589512), 'index': np.int64(2), 'embedding': None}
{'document': 'Tıp - İstanbul Üniversitesi', 'relevance_score': np.float32(0.215088), 'index': np.int64(0), 'embedding': None}
{'document': 'Hukuk - Ankara Üniversitesi', 'relevance_score': np.float32(-0.1418435), 'index': np.int64(1), 'embedding': None}


In [22]:
# Sorgu parser — doğal dil → yapılandırılmış JSON
PARSER_PROMPT = """Sen bir üniversite tercih asistanının sorgu ayrıştırıcısısın.
Kullanıcının doğal dildeki sorgusunu, veritabanı araması için yapılandırılmış JSON'a çevir.

MEVCUT FİLTRE ALANLARI:
- ilAdi: "İSTANBUL", "ANKARA", "İZMİR", "KOCAELİ", "BURSA" vb. (BÜYÜK HARF)
- universiteTuru: "DEVLET" veya "VAKIF"
- ogrenimDiliAdi: "Türkçe", "İngilizce", "İngilizce (%30)", "Almanca" vb.
- puanTuru: "SAY", "SÖZ", "EA", "DİL", "TYT"
- bursOraniAdi: "Burslu", "%50 İndirimli", "%25 İndirimli", "Ücretli"
- birimTuruAdi: "LISANS" (4 yıllık) veya "ÖNLISANS" (2 yıllık)
- basariSirasi_max: sayı (üst sınır — "300 bin sıralamayla" veya "ilk 10 bine giren")
- puan_min: sayı (kullanıcının aldığı puan — "450 aldım", "480 puanla girebileceğim")
- ucret_max: sayı (TL)

ÇIKTI FORMATI (JSON):
{
  "filtreler": { alan: değer, ... },
  "arama_metni": "vektör aramada kullanılacak serbest metin"
}

KURALLAR:
- Sadece sorguda AÇIKÇA geçen filtreleri ekle
- Şehir isimlerini BÜYÜK HARF yaz
- Bölüm adları (tıp, mühendislik, hukuk vb.) filtre DEĞİL, arama_metni'ne yaz
- Bilinmeyen alan uydurma

AKILLI ÇIKARIM (birimTuruAdi için):
- "tıp", "hukuk", "mühendislik", "eczacılık", "diş hekimliği", "mimarlık", "psikoloji" → LISANS
- "teknikerlik", "operatörlük", "yardımcılığı" → ÖNLISANS

ÖRNEKLER:

Sorgu: "İstanbul'daki tıp fakülteleri"
Çıktı: {"filtreler": {"ilAdi": "İSTANBUL", "birimTuruAdi": "LISANS"}, "arama_metni": "tıp"}

Sorgu: "450 puan aldım hangi mühendislik bölümlerine girebilirim"
Çıktı: {"filtreler": {"puan_min": 450, "birimTuruAdi": "LISANS"}, "arama_metni": "mühendislik"}

Sorgu: "İstanbul'da İngilizce bilgisayar mühendisliği"
Çıktı: {"filtreler": {"ilAdi": "İSTANBUL", "ogrenimDiliAdi": "İngilizce"}, "arama_metni": "bilgisayar mühendisliği"}

Sorgu: "300 bin sıralamayla mühendislik"
Çıktı: {"filtreler": {"basariSirasi_max": 300000}, "arama_metni": "mühendislik"}

Sorgu: "Yaratıcılığımı kullanabileceğim bölümler"
Çıktı: {"filtreler": {}, "arama_metni": "yaratıcılık tasarım sanat mimarlık"}

Şimdi sıradaki sorguyu ayrıştır:

Sorgu: "{kullanici_sorgusu}"
Çıktı:"""


def sorguyu_parse_et(kullanici_sorgusu: str) -> dict | None:
    prompt = PARSER_PROMPT.replace("{kullanici_sorgusu}", kullanici_sorgusu)
    try:
        r = openai_client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        return json.loads(r.choices[0].message.content)
    except Exception as e:
        print(f"Parse hatası: {e}")
        return None

In [23]:
# Parser'ı çağır, dönen JSON'u gör
sorgu = "İstanbul'da 450 puanla İngilizce bilgisayar mühendisliği"
parsEdilenSorgu = sorguyu_parse_et(sorgu)

print("Sorgu:", sorgu)
print("\nDönüş tipi:", type(parsEdilenSorgu).__name__)
print("Anahtarlar:", list(parsEdilenSorgu.keys()))
print("\nParse edilen sorgunun JSON çıktısı:")
print(json.dumps(parsEdilenSorgu, ensure_ascii=False, indent=2))

Sorgu: İstanbul'da 450 puanla İngilizce bilgisayar mühendisliği

Dönüş tipi: dict
Anahtarlar: ['filtreler', 'arama_metni']

Parse edilen sorgunun JSON çıktısı:
{
  "filtreler": {
    "ilAdi": "İSTANBUL",
    "puan_min": 450,
    "ogrenimDiliAdi": "İngilizce",
    "birimTuruAdi": "LISANS"
  },
  "arama_metni": "bilgisayar mühendisliği"
}


In [24]:
# Parse dict'inden Qdrant Filter nesnesi üret
def filtre_olustur(filtreler_dict: dict) -> Filter | None:
    if not filtreler_dict:
        return None

    kosullar = []
    for alan, deger in filtreler_dict.items():
        if alan == "basariSirasi_max":
            kosullar.append(FieldCondition(key="basariSirasi", range=Range(lte=deger)))
        elif alan == "puan_min":
            kosullar.append(FieldCondition(key="minPuan", range=Range(lte=deger)))
        elif alan == "ucret_max":
            kosullar.append(FieldCondition(key="ucret", range=Range(lte=deger)))
        else:
            kosullar.append(FieldCondition(key=alan, match=MatchValue(value=deger)))

    return Filter(must=kosullar) if kosullar else None

In [25]:
# parse_sonucu['filtreler'] alt sözlüğünü filtre_olustur'a geçir
qdrant_filter = filtre_olustur(parsEdilenSorgu['filtreler'])

print("Girdi (parse_sonucu['filtreler']):",parsEdilenSorgu['filtreler'])
print(f"\nDönen tip: {type(qdrant_filter).__name__}")
print(f"Koşul sayısı: {len(qdrant_filter.must)}")
print(f"Her koşulun içi:")
for i, k in enumerate(qdrant_filter.must, 1):
    print(f"  {i}. key={k.key}")
    if k.match: print(f"     match: {k.match.value}")
    if k.range: print(f"     range: lte={k.range.lte}, gte={k.range.gte}")

print("\nHam Qdrant nesnesi:\n",qdrant_filter)

Girdi (parse_sonucu['filtreler']): {'ilAdi': 'İSTANBUL', 'puan_min': 450, 'ogrenimDiliAdi': 'İngilizce', 'birimTuruAdi': 'LISANS'}

Dönen tip: Filter
Koşul sayısı: 4
Her koşulun içi:
  1. key=ilAdi
     match: İSTANBUL
  2. key=minPuan
     range: lte=450.0, gte=None
  3. key=ogrenimDiliAdi
     match: İngilizce
  4. key=birimTuruAdi
     match: LISANS

Ham Qdrant nesnesi:
 should=None min_should=None must=[FieldCondition(key='ilAdi', match=MatchValue(value='İSTANBUL'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='minPuan', match=None, range=Range(lt=None, gt=None, gte=None, lte=450.0), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='ogrenimDiliAdi', match=MatchValue(value='İngilizce'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldConditi

In [26]:
# Fusion adaylarını Jina reranker ile yeniden sırala
def rerank_et(arama_metni: str, sonuclar, top_k: int = 5):
    if not sonuclar:
        return sonuclar

    dokumanlar = []
    for p in sonuclar:
        pl = p.payload
        metin = f"{pl.get('birimAdi','')} - {pl.get('universiteAdi','')} - {pl.get('ilAdi','')} - {pl.get('ogrenimDiliAdi','')}"
        dokumanlar.append(metin)

    sonuc = reranker.rerank(arama_metni, dokumanlar, top_n=top_k)

    yeniden_sirali = []
    for r in sonuc:
        orijinal = sonuclar[r['index']]
        orijinal.score = float(r['relevance_score'])   # ← var olan score alanına yaz
        yeniden_sirali.append(orijinal)

    return yeniden_sirali

In [27]:
# rerank_et kanıtı — fusion çıktısı vs rerank çıktısı
sorgu = "İstanbul'da İngilizce tıp"
parse = sorguyu_parse_et(sorgu)
qfilter = filtre_olustur(parse['filtreler'])
q_vek = model.encode(parse['arama_metni'], normalize_embeddings=True).tolist()

ham = qdrant.query_points(
    collection_name=KOLEKSIYON,
    prefetch=[
        Prefetch(query=q_vek, using="title", limit=30, filter=qfilter),
        Prefetch(query=q_vek, using="content", limit=30, filter=qfilter),
        Prefetch(query=Document(text=parse['arama_metni'], model="Qdrant/bm25"),
                 using="bm25", limit=30, filter=qfilter),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=10,
    with_payload=["universiteAdi", "birimAdi", "ilAdi", "ogrenimDiliAdi"],
).points

print(f"Sorgu: {sorgu!r}\n")
print("── FUSION sırası (rerank YOK) ──")
for i, p in enumerate(ham, 1):
    pl = p.payload
    print(f"  {i}. [{p.score:.3f}] {pl['birimAdi']} — {pl['universiteAdi']} ({pl.get('ogrenimDiliAdi')})")

# Not: rerank_et 'ham' listesini yerinde değiştirir (score alanını ezer),
# o yüzden fusion skorlarını yukarıda zaten yazdırdık
reranked = rerank_et(parse['arama_metni'], ham, top_k=10)
print("\n── JINA RERANK sırası ──")
for i, p in enumerate(reranked, 1):
    pl = p.payload
    print(f"  {i}. [{p.score:.3f}] {pl['birimAdi']} — {pl['universiteAdi']} ({pl.get('ogrenimDiliAdi')})")

Sorgu: "İstanbul'da İngilizce tıp"

── FUSION sırası (rerank YOK) ──
  1. [1.000] Tıp (İngilizce) (%50 İndirimli) — İSTANBUL SAĞLIK VE TEKNOLOJİ ÜNİVERSİTESİ (İngilizce)
  2. [0.795] Tıp (İngilizce) — İSTANBUL ÜNİVERSİTESİ (İngilizce)
  3. [0.658] Tıp (İngilizce) (KKTC Uyruklu) — İSTANBUL ÜNİVERSİTESİ (İngilizce)
  4. [0.524] Tıp (İngilizce) (%50 İndirimli) — MALTEPE ÜNİVERSİTESİ (İSTANBUL) (İngilizce)
  5. [0.444] Tıp (İngilizce) (Burslu) — İSTANBUL SAĞLIK VE TEKNOLOJİ ÜNİVERSİTESİ (İngilizce)
  6. [0.386] Tıp (İngilizce) (%50 İndirimli) — İSTANBUL AYDIN ÜNİVERSİTESİ (İngilizce)
  7. [0.337] Tıp (İngilizce) (Burslu) — BAHÇEŞEHİR ÜNİVERSİTESİ (İSTANBUL) (İngilizce)
  8. [0.301] Tıp (İngilizce) — İSTANBUL ÜNİVERSİTESİ-CERRAHPAŞA (İngilizce)
  9. [0.292] Tıp (İngilizce) (Burslu) — İSTANBUL OKAN ÜNİVERSİTESİ (İngilizce)
  10. [0.291] Tıp (İngilizce) (%50 İndirimli) — KOÇ ÜNİVERSİTESİ (İSTANBUL) (İngilizce)

── JINA RERANK sırası ──
  1. [0.200] Tıp (İngilizce) (%50 İndirimli) — İSTANBUL S

In [28]:
# 3 kanal fusion + Jina reranking: geniş havuz çek, listwise süz
def arama_yap(kullanici_sorgusu: str, limit: int = 5, aday_sayisi: int = 20):
    parse = sorguyu_parse_et(kullanici_sorgusu)
    if parse is None:
        return None, None

    qfilter = filtre_olustur(parse['filtreler'])
    q_vek = model.encode(parse['arama_metni'], normalize_embeddings=True).tolist()

    sonuclar = qdrant.query_points(
        collection_name=KOLEKSIYON,
        prefetch=[
            Prefetch(query=q_vek, using="title", limit=30, filter=qfilter),
            Prefetch(query=q_vek, using="content", limit=30, filter=qfilter),
            Prefetch(
                query=Document(text=parse['arama_metni'], model="Qdrant/bm25"),
                using="bm25", limit=30, filter=qfilter,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=aday_sayisi,   # fusion'dan geniş aday havuzu (rerank için)
        with_payload=[
            "universiteAdi", "birimAdi", "ilAdi",
            "ogrenimDiliAdi", "puanTuru", "universiteTuru",
            "bursOraniAdi", "ucret",
            "minPuan", "basariSirasi",
            "minPuan1", "basariSirasi1",
            "minPuan2", "basariSirasi2",
            "minPuan3", "basariSirasi3",
        ],
    )

    # Reranking: fusion adaylarını Jina ile yeniden sırala, en iyi 'limit' kadarını al
    rerank_sonuc = rerank_et(parse['arama_metni'], sonuclar.points, top_k=limit)
    return rerank_sonuc, parse

In [29]:
# arama_yap: parse + filter + 3-kanal + rerank hepsini içinde yapıyor
test_sorgu = "İstanbul'da tıp okumak istiyorum"
sonuclar, parse = arama_yap(test_sorgu, limit=5)

print(f"Sorgu: {test_sorgu!r}")
print(f"Parse (arama_yap içinden): {json.dumps(parse, ensure_ascii=False)}")
print(f"\nDönen sonuç sayısı: {len(sonuclar)}")
print(f"İlk sonucun tipi: {type(sonuclar[0]).__name__}")
print(f"\nHer sonucun özet (rerank sonrası sıra):")
for i, p in enumerate(sonuclar, 1):
    pl = p.payload
    rr = getattr(p, 'rerank_skoru', None)
    rr_str = f"rerank={rr:.3f}" if rr is not None else "rerank=-"
    print(f"  {i}. [{rr_str}] {pl['universiteAdi']} — {pl['birimAdi']}")
    print(f"     il={pl['ilAdi']}, dil={pl.get('ogrenimDiliAdi')}, sıra={pl.get('basariSirasi')}")


Sorgu: "İstanbul'da tıp okumak istiyorum"
Parse (arama_yap içinden): {"filtreler": {"ilAdi": "İSTANBUL", "birimTuruAdi": "LISANS"}, "arama_metni": "tıp"}

Dönen sonuç sayısı: 5
İlk sonucun tipi: ScoredPoint

Her sonucun özet (rerank sonrası sıra):
  1. [rerank=-] İSTANBUL ÜNİVERSİTESİ — Tıp
     il=İSTANBUL, dil=Türkçe, sıra=4532.0
  2. [rerank=-] İSTANBUL ÜNİVERSİTESİ — Tıp (İngilizce)
     il=İSTANBUL, dil=İngilizce, sıra=3456.0
  3. [rerank=-] SAĞLIK BİLİMLERİ ÜNİVERSİTESİ (İSTANBUL) — Tıp
     il=İSTANBUL, dil=Türkçe, sıra=7346.0
  4. [rerank=-] İSTANBUL ÜNİVERSİTESİ-CERRAHPAŞA — Tıp
     il=İSTANBUL, dil=Türkçe, sıra=3082.0
  5. [rerank=-] İSTANBUL MEDENİYET ÜNİVERSİTESİ — Tıp
     il=İSTANBUL, dil=Türkçe, sıra=9787.0


In [30]:
# Ham sonuçları okunabilir metne çevir (Python, deterministik)
def programlari_formatla(sonuclar, hedef_sira=None, hedef_puan=None):
    satirlar = ["SİZİN İÇİN BULUNAN PROGRAMLAR:\n"]
    for i, p in enumerate(sonuclar, 1):
        pl = p.payload
        parcalar = [pl.get('ilAdi', '?')]
        if pl.get('ogrenimDiliAdi'):
            parcalar.append(pl['ogrenimDiliAdi'])
        if pl.get('universiteTuru'):
            parcalar.append(pl['universiteTuru'])
        if pl.get('puanTuru'):
            parcalar.append(f"{pl['puanTuru']} puan")

        sira = pl.get('basariSirasi')
        puan = pl.get('minPuan')
        if sira:
            parcalar.append(f"sıra: {int(sira)}")
        if puan:
            parcalar.append(f"min puan: {puan:.2f}")

        if hedef_sira and sira:
            s = int(sira)
            if s <= hedef_sira:
                parcalar.append("[GİRİLEBİLİR ✓]")
            elif s <= hedef_sira * 1.10:
                parcalar.append("[SINIRDA ⚠]")
            else:
                parcalar.append("[GİRİLEMEZ ✗]")

        if hedef_puan and puan:
            if puan <= hedef_puan:
                parcalar.append("[PUAN YETERLİ ✓]")
            elif puan <= hedef_puan + 10:
                parcalar.append("[PUAN SINIRDA ⚠]")
            else:
                parcalar.append("[PUAN YETMEZ ✗]")

        detay = " | ".join(parcalar)
        satirlar.append(f"{i}. {pl.get('universiteAdi', '?')} — {pl.get('birimAdi', '?')}")
        satirlar.append(f"   {detay}")

        trend_puan = [pl.get('minPuan'), pl.get('minPuan1'), pl.get('minPuan2'), pl.get('minPuan3')]
        trend_sira = [pl.get('basariSirasi'), pl.get('basariSirasi1'), pl.get('basariSirasi2'), pl.get('basariSirasi3')]
        if any(trend_puan):
            puan_str = " → ".join(f"{p:.1f}" if p else "-" for p in trend_puan)
            sira_str = " → ".join(f"{int(s)}" if s else "-" for s in trend_sira)
            satirlar.append(f"   Son 4 yıl puan: {puan_str}")
            satirlar.append(f"   Son 4 yıl sıra: {sira_str}")
        satirlar.append("")

    return "\n".join(satirlar)

In [31]:
# Bir önceki hücreden gelen 'sonuclar' ve 'parse' ile formatla
hedef_puan = parse['filtreler'].get('puan_min')
hedef_sira = parse['filtreler'].get('basariSirasi_max')

print(f"Hedef puan: {hedef_puan}, Hedef sıra: {hedef_sira}")
print("─" * 60)
metin = programlari_formatla(sonuclar, hedef_sira=hedef_sira, hedef_puan=hedef_puan)
print(metin)
print("─" * 60)


Hedef puan: None, Hedef sıra: None
────────────────────────────────────────────────────────────
SİZİN İÇİN BULUNAN PROGRAMLAR:

1. İSTANBUL ÜNİVERSİTESİ — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 4532 | min puan: 518.33
   Son 4 yıl puan: 518.3 → 516.6 → 527.1 → 526.9
   Son 4 yıl sıra: 4532 → 5145 → 5090 → 4030

2. İSTANBUL ÜNİVERSİTESİ — Tıp (İngilizce)
   İSTANBUL | İngilizce | DEVLET | SAY puan | sıra: 3456 | min puan: 522.49
   Son 4 yıl puan: 522.5 → 522.7 → 532.3 → 531.6
   Son 4 yıl sıra: 3456 → 3689 → 3429 → 2712

3. SAĞLIK BİLİMLERİ ÜNİVERSİTESİ (İSTANBUL) — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 7346 | min puan: 509.20
   Son 4 yıl puan: 509.2 → 506.7 → 518.8 → 516.9
   Son 4 yıl sıra: 7346 → 8071 → 8437 → 8079

4. İSTANBUL ÜNİVERSİTESİ-CERRAHPAŞA — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 3082 | min puan: 524.07
   Son 4 yıl puan: 524.1 → 524.6 → 533.0 → 532.2
   Son 4 yıl sıra: 3082 → 3294 → 3212 → 2559

5. İSTANBUL MEDENİYET ÜNİVERSİT

In [32]:
# GPT ile kısa yorum ekle
CEVAP_PROMPT = """Sen bir üniversite tercih danışmanısın. Kullanıcının sorusu ve
sistemin bulduğu programlar aşağıda. 3-4 cümlelik kısa, samimi bir yorum yaz.
Sayıları/isimleri değiştirme, verilen listeye sadık kal.

Kullanıcı sorusu: {soru}

Bulunan programlar:
{programlar}

Kısa yorum:"""


def cevap_uret(kullanici_sorgusu: str) -> str:
    sonuclar, parse = arama_yap(kullanici_sorgusu, limit=5)
    if sonuclar is None:
        return "Sorgunuzu anlayamadım."

    hedef_sira = parse['filtreler'].get('basariSirasi_max') if parse else None
    hedef_puan = parse['filtreler'].get('puan_min') if parse else None
    program_metni = programlari_formatla(sonuclar, hedef_sira=hedef_sira, hedef_puan=hedef_puan)

    prompt = CEVAP_PROMPT.replace("{soru}", kullanici_sorgusu).replace("{programlar}", program_metni)
    r = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=250,
    )
    yorum = r.choices[0].message.content

    return f"{program_metni}\n\n{yorum}"

In [33]:
# cevap_uret: tüm adımları tek fonksiyona indirir (arama + formatla + GPT yorum)
print(cevap_uret("İstanbul'da 450 puanla İngilizce bilgisayar mühendisliği"))


SİZİN İÇİN BULUNAN PROGRAMLAR:

1. BAHÇEŞEHİR ÜNİVERSİTESİ (İSTANBUL) — Bilgisayar Mühendisliği (İngilizce) (%50 İndirimli)
   İSTANBUL | İngilizce | VAKIF | SAY puan | sıra: 178136 | min puan: 348.04 | [PUAN YETERLİ ✓]
   Son 4 yıl puan: 348.0 → 406.7 → 473.6 → 480.3
   Son 4 yıl sıra: 178136 → 75701 → 37603 → 31540

2. MEF ÜNİVERSİTESİ (İSTANBUL) — Bilgisayar Mühendisliği (İngilizce) (%50 İndirimli)
   İSTANBUL | İngilizce | VAKIF | SAY puan | sıra: 262156 | min puan: 313.86 | [PUAN YETERLİ ✓]
   Son 4 yıl puan: 313.9 → 361.4 → 415.0 → 440.2
   Son 4 yıl sıra: 262156 → 130084 → 92956 → 65433

3. İSTANBUL KÜLTÜR ÜNİVERSİTESİ — Bilgisayar Mühendisliği (İngilizce) (Ücretli)
   İSTANBUL | İngilizce | VAKIF | SAY puan | sıra: 251706 | min puan: 317.35 | [PUAN YETERLİ ✓]
   Son 4 yıl puan: 317.3 → - → 343.3 → 342.2
   Son 4 yıl sıra: 251706 → - → 202735 → 190317

4. BİRUNİ ÜNİVERSİTESİ (İSTANBUL) — Bilgisayar Mühendisliği (İngilizce) (Burslu)
   İSTANBUL | İngilizce | VAKIF | SAY puan | sı

In [34]:
# Deneme
sorgu = "İstanbul'da tıp okumak istiyorum"
print(cevap_uret(sorgu))

SİZİN İÇİN BULUNAN PROGRAMLAR:

1. İSTANBUL ÜNİVERSİTESİ — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 4532 | min puan: 518.33
   Son 4 yıl puan: 518.3 → 516.6 → 527.1 → 526.9
   Son 4 yıl sıra: 4532 → 5145 → 5090 → 4030

2. İSTANBUL ÜNİVERSİTESİ — Tıp (İngilizce)
   İSTANBUL | İngilizce | DEVLET | SAY puan | sıra: 3456 | min puan: 522.49
   Son 4 yıl puan: 522.5 → 522.7 → 532.3 → 531.6
   Son 4 yıl sıra: 3456 → 3689 → 3429 → 2712

3. SAĞLIK BİLİMLERİ ÜNİVERSİTESİ (İSTANBUL) — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 7346 | min puan: 509.20
   Son 4 yıl puan: 509.2 → 506.7 → 518.8 → 516.9
   Son 4 yıl sıra: 7346 → 8071 → 8437 → 8079

4. İSTANBUL ÜNİVERSİTESİ-CERRAHPAŞA — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 3082 | min puan: 524.07
   Son 4 yıl puan: 524.1 → 524.6 → 533.0 → 532.2
   Son 4 yıl sıra: 3082 → 3294 → 3212 → 2559

5. İSTANBUL MEDENİYET ÜNİVERSİTESİ — Tıp
   İSTANBUL | Türkçe | DEVLET | SAY puan | sıra: 9787 | min puan: 502.98
   Son 4 yıl 

In [ ]:
# En basit arayüz — sorgu kutusu + buton + çıktı
import gradio as gr

def asistan_cevap(sorgu):
    if not sorgu.strip():
        return "Lütfen bir soru yazın."
    return cevap_uret(sorgu)

arayuz = gr.Interface(
    fn=asistan_cevap,
    inputs=gr.Textbox(
        label="Sorunuz",
        placeholder="Örn: İstanbul'da İngilizce tıp okumak istiyorum",
        lines=2,
    ),
    outputs=gr.Textbox(label="Tercih Asistanı Cevabı", lines=20),
    title="YKS Tercih Asistanı",
    description="Doğal dille sorun, size uygun üniversite programlarını bulalım.",
    examples=[
        "İstanbul'da tıp okumak istiyorum",
        "450 puanla İngilizce bilgisayar mühendisliği",
        "300 bin sıralamayla yapay zeka bölümleri",
        "Ankara'da devlet üniversitesinde psikoloji",
    ],
)

arayuz.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
